# Grouping Data with Pandas

This notebook demonstrates how to group data using `pd.groupby()`. We will explore splitting the data into groups based on criteria, applying functions to each group independently (aggregation, transformation, filtering), and then combining the results into a data structure (Split-Apply-Combine pattern).

In [1]:
import pandas as pd
import numpy as np

### Splitting Data

Let's start by looking at census data, keeping only county-level data. 

In [2]:
df = pd.read_csv("datasets/census.csv")

df = df[df["SUMLEV"] == 50]

df.head()

,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,...,RDOMESTICMIG2011,RDOMESTICMIG2012,RDOMESTICMIG2013,RDOMESTICMIG2014,RDOMESTICMIG2015,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015
1,50,3,6,1,1,Alabama,Autauga County,54571,54571,54660,...,7.242091,-2.915927,-3.012349,2.265971,-2.530799,7.606016,-2.626146,-2.722002,2.592270,-2.187333
2,50,3,6,1,3,Alabama,Baldwin County,182265,182265,183193,...,14.832960,17.647293,21.845705,19.243287,17.197872,15.844176,18.559627,22.727626,20.317142,18.293499
3,50,3,6,1,5,Alabama,Barbour County,27457,27457,27341,...,-4.728132,-2.500690,-7.056824,-3.904217,-10.543299,-4.874741,-2.758113,-7.167664,-3.978583,-10.543299
4,50,3,6,1,7,Alabama,Bibb County,22915,22919,22861,...,-5.527043,-5.068871,-6.201001,-0.177537,0.177258,-5.088389,-4.363636,-5.403729,0.754533,1.107861
5,50,3,6,1,9,Alabama,Blount County,57322,57322,57373,...,1.807375,-1.177622,-1.748766,-2.062535,-1.369970,1.859511,-0.848580,-1.402476,-1.577232,-0.884411


If we wanted to calculate the average population constraint for each state, we could iterate through unique state names, filter the dataframe, and calculate the average. This approach is slow and not idiomatic pandas.

In [3]:
for state in df["STNAME"].unique():
    avg = np.average(df.where(df["STNAME"] == state).dropna()["CENSUS2010POP"])
    print("Average POP: " + state + ": "+ str(avg))

Average POP: Alabama: 71339.34328358209
Average POP: Alaska: 24490.724137931036
Average POP: Arizona: 426134.4666666667
Average POP: Arkansas: 38878.90666666667
Average POP: California: 642309.5862068966
Average POP: Colorado: 78581.1875
Average POP: Connecticut: 446762.125
Average POP: Delaware: 299311.3333333333
Average POP: District of Columbia: 601723.0
Average POP: Florida: 280616.5671641791
Average POP: Georgia: 60928.63522012578
Average POP: Hawaii: 272060.2
Average POP: Idaho: 35626.86363636364
Average POP: Illinois: 125790.50980392157
Average POP: Indiana: 70476.10869565218
Average POP: Iowa: 30771.262626262625
Average POP: Kansas: 27172.55238095238
Average POP: Kentucky: 36161.39166666667
Average POP: Louisiana: 70833.9375
Average POP: Maine: 83022.5625
Average POP: Maryland: 240564.66666666666
Average POP: Massachusetts: 467687.78571428574
Average POP: Michigan: 119080.0
Average POP: Minnesota: 60964.65517241379
Average POP: Mississippi: 36186.54878048781
Average POP: Missou

Instead, we can use `groupby()`. This function splits the dataframe into a `GroupBy` object. Iterating over the `GroupBy` object yields tuples of `(group_name, DataFrame_for_group)`.

In [4]:
# Group by method
for group, frame in df.groupby("STNAME"):
    avg = np.average(frame["CENSUS2010POP"])

    print("Average POP: " + group + ": "+ str(avg))

Average POP: Alabama: 71339.34328358209
Average POP: Alaska: 24490.724137931036
Average POP: Arizona: 426134.4666666667
Average POP: Arkansas: 38878.90666666667
Average POP: California: 642309.5862068966
Average POP: Colorado: 78581.1875
Average POP: Connecticut: 446762.125
Average POP: Delaware: 299311.3333333333
Average POP: District of Columbia: 601723.0
Average POP: Florida: 280616.5671641791
Average POP: Georgia: 60928.63522012578
Average POP: Hawaii: 272060.2
Average POP: Idaho: 35626.86363636364
Average POP: Illinois: 125790.50980392157
Average POP: Indiana: 70476.10869565218
Average POP: Iowa: 30771.262626262625
Average POP: Kansas: 27172.55238095238
Average POP: Kentucky: 36161.39166666667
Average POP: Louisiana: 70833.9375
Average POP: Maine: 83022.5625
Average POP: Maryland: 240564.66666666666
Average POP: Massachusetts: 467687.78571428574
Average POP: Michigan: 119080.0
Average POP: Minnesota: 60964.65517241379
Average POP: Mississippi: 36186.54878048781
Average POP: Missou

We can also pass a custom function to `groupby`. The function is evaluated on the DataFrame's index. Let's create a custom function to group states into batches based on their first letter.

In [5]:
df = df.set_index("STNAME")

def setBatchNum(item):
    if item[0] < "M":
        return 0
    elif item[0] < "Q":
        return 1
    else:
        return 2
    
for group, frame in df.groupby(setBatchNum):
    print(f"There are {str(len(frame))} records in group {str(group)}")

There are 1177 records in group 0
There are 1134 records in group 1
There are 831 records in group 2


### Multiple Iteration and Grouping on Hierarchy

Let's load an Airbnb dataset to see more advanced `groupby` features.

In [6]:
df = pd.read_csv("datasets/listings.csv")

df.head()

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,review_scores_value,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month
0,12147973,https://www.airbnb.com/rooms/12147973,20160906204935,2016-09-07,Sunny Bungalow in the City,"Cozy, sunny, family home. Master bedroom high...",The house has an open and cozy feel at the sam...,"Cozy, sunny, family home. Master bedroom high...",none,"Roslindale is quiet, convenient and friendly. ...",...,NaN,f,NaN,NaN,f,moderate,f,f,1,NaN
1,3075044,https://www.airbnb.com/rooms/3075044,20160906204935,2016-09-07,Charming room in pet friendly apt,Charming and quiet room in a second floor 1910...,Small but cozy and quite room with a full size...,Charming and quiet room in a second floor 1910...,none,"The room is in Roslindale, a diverse and prima...",...,9.0,f,NaN,NaN,t,moderate,f,f,1,1.30
2,6976,https://www.airbnb.com/rooms/6976,20160906204935,2016-09-07,Mexican Folk Art Haven in Boston,"Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...",none,The LOCATION: Roslindale is a safe and diverse...,...,10.0,f,NaN,NaN,f,moderate,t,f,1,0.47
3,1436513,https://www.airbnb.com/rooms/1436513,20160906204935,2016-09-07,Spacious Sunny Bedroom Suite in Historic Home,Come experience the comforts of home away from...,Most places you find in Boston are small howev...,Come experience the comforts of home away from...,none,Roslindale is a lovely little neighborhood loc...,...,10.0,f,NaN,NaN,f,moderate,f,f,1,1.00
4,7651065,https://www.airbnb.com/rooms/7651065,20160906204935,2016-09-07,Come Home to Boston,"My comfy, clean and relaxing home is one block...","Clean, attractive, private room, one block fro...","My comfy, clean and relaxing home is one block...",none,"I love the proximity to downtown, the neighbor...",...,10.0,f,NaN,NaN,f,flexible,f,f,1,2.25


If we set our index to a MultiIndex (e.g., `cancellation_policy` and `review_scores_value`), we can group by multiple levels simultaneously by passing `level=(0, 1)`.

In [7]:
df=df.set_index(["cancellation_policy","review_scores_value"])

for group, frame in df.groupby(level=(0,1)):
    print(group)

('flexible', np.float64(2.0))
('flexible', np.float64(4.0))
('flexible', np.float64(5.0))
('flexible', np.float64(6.0))
('flexible', np.float64(7.0))
('flexible', np.float64(8.0))
('flexible', np.float64(9.0))
('flexible', np.float64(10.0))
('moderate', np.float64(2.0))
('moderate', np.float64(4.0))
('moderate', np.float64(6.0))
('moderate', np.float64(7.0))
('moderate', np.float64(8.0))
('moderate', np.float64(9.0))
('moderate', np.float64(10.0))
('strict', np.float64(2.0))
('strict', np.float64(3.0))
('strict', np.float64(4.0))
('strict', np.float64(5.0))
('strict', np.float64(6.0))
('strict', np.float64(7.0))
('strict', np.float64(8.0))
('strict', np.float64(9.0))
('strict', np.float64(10.0))
('super_strict_30', np.float64(6.0))
('super_strict_30', np.float64(7.0))
('super_strict_30', np.float64(8.0))
('super_strict_30', np.float64(9.0))
('super_strict_30', np.float64(10.0))


We can define a grouping function that looks at both index levels and groups them logically (e.g. treating instances of 10.0 review score separately from all other values).

In [8]:
def grouping(item):
    if item[1] == 10.0:
        return (item[0], "10.0")
    else:
        return (item[0], "not 10.0")
    
for group, frame in df.groupby(by=grouping):
    print(group)

('flexible', '10.0')
('flexible', 'not 10.0')
('moderate', '10.0')
('moderate', 'not 10.0')
('strict', '10.0')
('strict', 'not 10.0')
('super_strict_30', '10.0')
('super_strict_30', 'not 10.0')


### Aggregation: Split-Apply-Combine

The most common use case for `groupby` is aggregation using `.agg()`. This allows us to apply a summary function (like mean or sum) to each group.

In [9]:
df = df.reset_index()

df.groupby("cancellation_policy").agg({"review_scores_value": np.nanmean})

C:\Users\kanko\AppData\Local\Temp\ipykernel_16180\1193514434.py:3: FutureWarning: The provided callable <function nanmean at 0x0000022EC41E3EC0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby("cancellation_policy").agg({"review_scores_value": np.nanmean})


,review_scores_value
cancellation_policy,
flexible,9.237421
moderate,9.307398
strict,9.081441
super_strict_30,8.537313


We can pass a dictionary to `.agg()` where keys are columns and values are the functions to apply. We can even pass a tuple of functions to apply multiple functions to a single column.

In [10]:
df.groupby("cancellation_policy").agg({"review_scores_value":(np.nanmean,np.nanstd),
                                      "reviews_per_month":np.nanmean})

C:\Users\kanko\AppData\Local\Temp\ipykernel_16180\3943869473.py:1: FutureWarning: The provided callable <function nanmean at 0x0000022EC41E3EC0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby("cancellation_policy").agg({"review_scores_value":(np.nanmean,np.nanstd),
C:\Users\kanko\AppData\Local\Temp\ipykernel_16180\3943869473.py:1: FutureWarning: The provided callable <function nanstd at 0x0000022EC41F4900> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  df.groupby("cancellation_policy").agg({"review_scores_value":(np.nanmean,np.nanstd),
C:\Users\kanko\AppData\Local\Temp\ipykernel_16180\3943869473.py:1: FutureWarning: The provided callable <function nanmean at 0x0000022EC41E3EC0> is currently using SeriesGroupBy.mean. In a future

review_scores_value           reviews_per_month
                                nanmean    nanstd           nanmean
cancellation_policy                                                
flexible                       9.237421  1.096271          1.829210
moderate                       9.307398  0.859859          2.391922
strict                         9.081441  1.040531          1.873467
super_strict_30                8.537313  0.840785          0.340143

### Transformation

Unlike aggregation which returns a single summary value per group, transformation returns a DataFrame of the same size as the original group. The function is applied broadcasting across the entire group. Here we use `.transform('mean')`.

In [11]:
cols=['cancellation_policy','review_scores_value']

transform_df=df[cols].groupby('cancellation_policy').transform('mean') # 'mean' can be used instead of np.nanmean

transform_df.head()

,review_scores_value
0,9.307398
1,9.307398
2,9.307398
3,9.307398
4,9.237421


We can then merge this transformed series back to our original DataFrame since they share the same index.

In [12]:
transform_df.rename({'review_scores_value':'mean_review_scores'}, axis='columns', inplace=True)

df=df.merge(transform_df, left_index=True, right_index=True)

df.head()

,cancellation_policy,review_scores_value,id,listing_url,scrape_id,last_scraped,name,summary,space,description,...,review_scores_location,requires_license,license,jurisdiction_names,instant_bookable,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month,mean_review_scores
0,moderate,NaN,12147973,https://www.airbnb.com/rooms/12147973,20160906204935,2016-09-07,Sunny Bungalow in the City,"Cozy, sunny, family home. Master bedroom high...",The house has an open and cozy feel at the sam...,"Cozy, sunny, family home. Master bedroom high...",...,NaN,f,NaN,NaN,f,f,f,1,NaN,9.307398
1,moderate,9.0,3075044,https://www.airbnb.com/rooms/3075044,20160906204935,2016-09-07,Charming room in pet friendly apt,Charming and quiet room in a second floor 1910...,Small but cozy and quite room with a full size...,Charming and quiet room in a second floor 1910...,...,9.0,f,NaN,NaN,t,f,f,1,1.30,9.307398
2,moderate,10.0,6976,https://www.airbnb.com/rooms/6976,20160906204935,2016-09-07,Mexican Folk Art Haven in Boston,"Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...",...,9.0,f,NaN,NaN,f,t,f,1,0.47,9.307398
3,moderate,10.0,1436513,https://www.airbnb.com/rooms/1436513,20160906204935,2016-09-07,Spacious Sunny Bedroom Suite in Historic Home,Come experience the comforts of home away from...,Most places you find in Boston are small howev...,Come experience the comforts of home away from...,...,10.0,f,NaN,NaN,f,f,f,1,1.00,9.307398
4,flexible,10.0,7651065,https://www.airbnb.com/rooms/7651065,20160906204935,2016-09-07,Come Home to Boston,"My comfy, clean and relaxing home is one block...","Clean, attractive, private room, one block fro...","My comfy, clean and relaxing home is one block...",...,9.0,f,NaN,NaN,f,f,f,1,2.25,9.237421


This allows us to easily compute row-wise calculations against group-level statistics, such as find the difference between a listing's review score and the average score of all listings with the same cancellation policy.

In [13]:
df['mean_diff']=np.absolute(df['review_scores_value']-df['mean_review_scores'])

df['mean_diff'].head()

0         NaN
1    0.307398
2    0.692602
3    0.692602
4    0.762579
Name: mean_diff, dtype: float64

### Filtering

The `filter()` function takes a function that evaluates to True or False for the *entire* group. If it returns False, the group is dropped from the result. Here, we keep only groups whose average review score is greater than 9.2.

In [14]:
df.groupby('cancellation_policy').filter(lambda x: np.nanmean(x['review_scores_value'])>9.2)

,cancellation_policy,review_scores_value,id,listing_url,scrape_id,last_scraped,name,summary,space,description,...,requires_license,license,jurisdiction_names,instant_bookable,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month,mean_review_scores,mean_diff
0,moderate,NaN,12147973,https://www.airbnb.com/rooms/12147973,20160906204935,2016-09-07,Sunny Bungalow in the City,"Cozy, sunny, family home. Master bedroom high...",The house has an open and cozy feel at the sam...,"Cozy, sunny, family home. Master bedroom high...",...,f,NaN,NaN,f,f,f,1,NaN,9.307398,NaN
1,moderate,9.0,3075044,https://www.airbnb.com/rooms/3075044,20160906204935,2016-09-07,Charming room in pet friendly apt,Charming and quiet room in a second floor 1910...,Small but cozy and quite room with a full size...,Charming and quiet room in a second floor 1910...,...,f,NaN,NaN,t,f,f,1,1.30,9.307398,0.307398
2,moderate,10.0,6976,https://www.airbnb.com/rooms/6976,20160906204935,2016-09-07,Mexican Folk Art Haven in Boston,"Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...",...,f,NaN,NaN,f,t,f,1,0.47,9.307398,0.692602
3,moderate,10.0,1436513,https://www.airbnb.com/rooms/1436513,20160906204935,2016-09-07,Spacious Sunny Bedroom Suite in Historic Home,Come experience the comforts of home away from...,Most places you find in Boston are small howev...,Come experience the comforts of home away from...,...,f,NaN,NaN,f,f,f,1,1.00,9.307398,0.692602
4,flexible,10.0,7651065,https://www.airbnb.com/rooms/7651065,20160906204935,2016-09-07,Come Home to Boston,"My comfy, clean and relaxing home is one block...","Clean, attractive, private room, one block fro...","My comfy, clean and relaxing home is one block...",...,f,NaN,NaN,f,f,f,1,2.25,9.237421,0.762579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3576,flexible,NaN,14689681,https://www.airbnb.com/rooms/14689681,20160906204935,2016-09-07,Beautiful loft style bedroom with large bathroom,You'd be living on the top floor of a four sto...,NaN,You'd be living on the top floor of a four sto...,...,f,NaN,NaN,f,f,f,1,NaN,9.237421,NaN
3577,flexible,NaN,13750763,https://www.airbnb.com/rooms/13750763,20160906204935,2016-09-07,Comfortable Space in the Heart of Brookline,"Our place is close to Coolidge Corner, Allston...",This space consists of 2 Rooms and a private b...,"Our place is close to Coolidge Corner, Allston...",...,f,NaN,NaN,f,f,f,1,NaN,9.237421,NaN
3579,flexible,NaN,14852179,https://www.airbnb.com/rooms/14852179,20160906204935,2016-09-07,Spacious Queen Bed Room Close to Boston Univer...,- Grocery: A full-size Star market is 2 minute...,NaN,- Grocery: A full-size Star market is 2 minute...,...,f,NaN,NaN,f,f,f,1,NaN,9.237421,NaN
3582,flexible,NaN,14585486,https://www.airbnb.com/rooms/14585486,20160906204935,2016-09-07,Gorgeous funky apartment,Funky little apartment close to public transpo...,Modern and relaxed space with many facilities ...,Funky little apartment close to public transpo...,...,f,NaN,NaN,f,f,f,1,NaN,9.237421,NaN


### Applying Custom Functions

The `.apply()` function in a `groupby` object applies an arbitrary function to each group DataFrame. Let's start with a fresh DataFrame slice.

In [15]:
df=pd.read_csv("datasets/listings.csv")

df=df[['cancellation_policy','review_scores_value']]
df.head()

,cancellation_policy,review_scores_value
0,moderate,NaN
1,moderate,9.0
2,moderate,10.0
3,moderate,10.0
4,flexible,10.0


Now we define a function that calculates the mean review score *for the group*, creates a new column comparing each individual score to the group mean, and returns the modified DataFrame. Using `apply` simplifies operations that would take multiple steps (like we did with `transform`).

In [16]:
def calc_mean_review_scores(group):
    avg=np.nanmean(group["review_scores_value"])

    group["review_scores_mean"]=np.abs(avg-group["review_scores_value"])
    return group

df.groupby('cancellation_policy').apply(calc_mean_review_scores).head()

C:\Users\kanko\AppData\Local\Temp\ipykernel_16180\2570030050.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('cancellation_policy').apply(calc_mean_review_scores).head()


cancellation_policy  review_scores_value  \
cancellation_policy                                               
flexible            4             flexible                 10.0   
                    5             flexible                 10.0   
                    10            flexible                 10.0   
                    11            flexible                  9.0   
                    12            flexible                 10.0   

                        review_scores_mean  
cancellation_policy                         
flexible            4             0.762579  
                    5             0.762579  
                    10            0.762579  
                    11            0.237421  
                    12            0.762579